In [3]:
# ==========================================================
# NGBoost Pressure Gradient Model — Reviewer Prediction Script
# ==========================================================
# This script loads a pre-trained NGBoost model and generates
# mean predictions with 95% confidence intervals on new data.
# The model is NOT retrained here — it is loaded exactly as saved.
#
# Requirements (install with the exact versions noted below):
#   pip install ngboost==<VERSION> scikit-learn==<VERSION> pandas numpy openpyxl


import pickle
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. LOAD THE SAVED MODEL BUNDLE
# ----------------------------------------------------------
# Place this script in the SAME folder as the .pkl file, or
# update this path to point to it.
MODEL_PATH = "ngboost_pressure_gradient_model.pkl"

with open(MODEL_PATH, "rb") as f:
    model_bundle = pickle.load(f)

loaded_model = model_bundle["model"]
FEATURES = model_bundle["feature_names"]
LOG_CLIP_MIN, LOG_CLIP_MAX = model_bundle["log_clip_bounds"]
Z = model_bundle["ci_z_value"]

print("Model loaded successfully.")
print(f"Required input features (in this exact order): {FEATURES}")
print(f"Trained with ngboost=={model_bundle['training_info']['ngboost_version']}, "
      f"scikit-learn=={model_bundle['training_info']['sklearn_version']}")


# ----------------------------------------------------------
# 2. PREDICTION FUNCTION (mean + 95% CI, original units)
# ----------------------------------------------------------
def predict_with_uncertainty(X_new: pd.DataFrame) -> pd.DataFrame:
    """
    X_new : DataFrame containing at least the columns listed in FEATURES.
            Extra columns are ignored; order is enforced internally.

    Returns a DataFrame with:
        Predicted_Mean       - point prediction, original (untransformed) units
        Lower_Bound_95CI     - lower bound of 95% confidence interval
        Upper_Bound_95CI     - upper bound of 95% confidence interval
        Log_Standard_Deviation - model's uncertainty in log-space (for reference)
    """
    missing = [c for c in FEATURES if c not in X_new.columns]
    if missing:
        raise KeyError(f"Input data is missing required columns: {missing}")

    X_ordered = X_new[FEATURES].reset_index(drop=True)

    pred_log = loaded_model.predict(X_ordered)
    dist = loaded_model.pred_dist(X_ordered)
    log_std = dist.scale

    pred_mean = np.exp(np.clip(pred_log, LOG_CLIP_MIN, LOG_CLIP_MAX))
    lower_95 = np.exp(pred_log - Z * log_std)
    upper_95 = np.exp(pred_log + Z * log_std)

    return pd.DataFrame({ "Predicted_Mean": pred_mean,
        "Lower_Bound_95CI": lower_95,
        "Upper_Bound_95CI": upper_95,
        "Log_Standard_Deviation": log_std})


# ----------------------------------------------------------
# 3. EXAMPLE USAGE — replace with your own test data
# ----------------------------------------------------------
if __name__ == "__main__":
    # Option A: load your own data from an Excel file
    # test_data = pd.read_excel("your_test_data.xlsx", sheet_name="Sheet1")

    # Option B: build a small DataFrame by hand to test a single case
    test_data = pd.DataFrame({
        "Frtp":    [9.45],   # <- replace with real values
        "Beta":    [0.395],
        "Prtp":    [2.12],
        "Refo":    [374.344],
        "BoPh_Pf": [0.000281],   # note: BoPh_Pf = Bo * Ph_Pf — compute this upstream
        "Wefo":    [0.311366],
        "Sugo":    [2113184],})

    results = predict_with_uncertainty(test_data)
    print("\nPrediction results:")
    print(results)

Model loaded successfully.
Required input features (in this exact order): ['Frtp', 'Beta', 'Prtp', 'Refo', 'BoPh_Pf', 'Wefo', 'Sugo']
Trained with ngboost==0.5.11, scikit-learn==1.8.0

Prediction results:
   Predicted_Mean  Lower_Bound_95CI  Upper_Bound_95CI  Log_Standard_Deviation
0     2371.105947       1841.269304       3053.406364                0.129031
